In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

BASE = Path(r"C:/Users/janku/Documents/KCL/Research Project/Research Project")
EXCEL_PATH = BASE / "results" / "logs" / "gradient_boosting_results.xlsx"
FIG_DIR = BASE / "results" / "figures" / "gradboost_from_excel"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SHEETS = [
    "split_info",
    "train_test_distribution",
    "cv_folds",
    "cv_balance",
    "cv_summary",
    "subgroup_gender",
    "subgroup_age",
    "subgroup_gender_age",
    "fairness",
    "test_summary",
]


def save_fig(fig, name, dpi=200):
  path = FIG_DIR / name
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
  plt.close(fig)
  print(f"Saved: {path}")
  return path


def load_results(path=EXCEL_PATH):
  path = Path(path)
  if not path.exists():
    raise FileNotFoundError(f"Results file not found: {path}")
  data = {}
  xl = pd.ExcelFile(path)
  for sheet in SHEETS:
    if sheet in xl.sheet_names:
      data[sheet] = pd.read_excel(path, sheet_name=sheet)
    else:
      print(f"Warning: missing sheet '{sheet}'")
  return data


def enrich_model_columns(df):
  """Add dataset and short model labels from the 'model' column."""
  out = df.copy()
  if "model" not in out.columns:
    return out
  out["dataset"] = np.where(
    out["model"].str.contains("RADAR", case=False, na=False),
    "RADAR",
    np.where(out["model"].str.contains("Android", case=False, na=False), "Androids", "Other"),
  )
  out["model_short"] = (
    out["model"].str.replace("RADAR ", "", regex=False).str.replace("Androids ", "", regex=False)
  )
  out["model_label"] = out["dataset"] + " | " + out["model_short"]
  return out


def numeric_metric_cols(df, exclude=None):
  exclude = set(exclude or [])
  skip = {"model", "dataset", "model_short", "model_label", "fold", "group", "label", "matrix", "subset", "comparison", "test_name"}
  skip |= exclude
  cols = []
  for c in df.columns:
    if c in skip:
      continue
    if pd.api.types.is_numeric_dtype(df[c]):
      cols.append(c)
  return cols


def normalize_gender_token(value):
  """Map gender_0/1, 0/1, F/M, etc. to female/male."""
  if pd.isna(value):
    return value
  token = str(value).strip().lower()
  mapping = {
    "gender_0": "female",
    "gender_1": "male",
    "0": "female",
    "1": "male",
    "f": "female",
    "m": "male",
    "female": "female",
    "male": "male",
    "woman": "female",
    "man": "male",
  }
  return mapping.get(token, str(value).strip())


def normalize_subgroup_group_value(value):
  """Normalize pure gender labels and age x gender intersections."""
  if pd.isna(value):
    return value
  text = str(value).strip()
  parts = text.split()
  if len(parts) >= 2:
    age_part = parts[0]
    gender_part = " ".join(parts[1:])
    return f"{age_part} {normalize_gender_token(gender_part)}"
  return normalize_gender_token(text)


def combine_gender_subgroups(df, group_col="group"):
  """Unify gender labels and merge duplicate rows after normalization."""
  out = df.copy()
  out[group_col] = out[group_col].map(normalize_subgroup_group_value)

  id_cols = [c for c in ["model", "dataset", "model_short", "model_label", group_col] if c in out.columns]
  agg_dict = {}
  for col in out.columns:
    if col in id_cols:
      continue
    if col in {"n", "n_depressed", "n_control"}:
      agg_dict[col] = "sum"
    elif col in {"accuracy", "f1", "roc_auc"}:
      agg_dict[col] = "mean"

  if not agg_dict:
    return out.drop_duplicates(subset=id_cols)

  return out.groupby(id_cols, as_index=False).agg(agg_dict)


data = load_results()
for k, v in data.items():
  data[k] = enrich_model_columns(v)
  print(f"{k}: {v.shape} — models: {data[k]['model'].unique().tolist() if 'model' in data[k].columns else 'n/a'}")

split_info: (5, 5) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost']
train_test_distribution: (2, 4) — models: ['RADAR GBC']
cv_folds: (32, 30) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost', nan]
cv_balance: (10, 13) — models: ['RADAR GBR', 'RADAR GBC']
cv_summary: (10, 18) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost']
subgroup_gender: (10, 8) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost']
subgroup_age: (15, 8) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost']
subgroup_gender_age: (30, 8) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost']
fairness: (5, 3) — models: ['RADAR GBR', 'RADAR GBC', 'RADAR GPBoost', 'Androids GBC', 'Androids GPBoost']


In [2]:
# --- 1. Split info: train vs test size by model ---
split_df = data["split_info"]

melted = split_df.melt(
  id_vars=["model_label", "dataset", "model_short"],
  value_vars=["train_rows", "test_rows", "train_participants", "test_participants"],
  var_name="split_metric",
  value_name="count",
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=melted, x="model_label", y="count", hue="split_metric", ax=ax)
ax.set_title("Data split sizes by model")
ax.set_xlabel("Model")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=35)
plt.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
save_fig(fig, "01_split_info.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\01_split_info.png


WindowsPath('C:/Users/janku/Documents/KCL/Research Project/Research Project/results/figures/gradboost_from_excel/01_split_info.png')

In [3]:
# --- 2. Train/test label distribution ---
dist_df = data["train_test_distribution"]

# Expect columns: model, label, train, test (or similar)
value_cols = [c for c in dist_df.columns if c.lower() in {"train", "test"}]
if not value_cols:
  value_cols = [c for c in dist_df.columns if c not in {"model", "dataset", "model_short", "model_label", "label"} and pd.api.types.is_numeric_dtype(dist_df[c])]

melted = dist_df.melt(
  id_vars=["model_label", "dataset", "label"],
  value_vars=value_cols,
  var_name="split",
  value_name="count",
)

g = sns.catplot(
  data=melted,
  kind="bar",
  x="label",
  y="count",
  hue="split",
  col="dataset",
  col_wrap=2,
  height=4,
  aspect=1.2,
  sharey=False,
)
g.fig.suptitle("Train vs test class distribution", y=1.02)
g.set_axis_labels("Class", "Count")
save_fig(g.fig, "02_train_test_distribution_by_dataset.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\02_train_test_distribution_by_dataset.png


WindowsPath('C:/Users/janku/Documents/KCL/Research Project/Research Project/results/figures/gradboost_from_excel/02_train_test_distribution_by_dataset.png')

In [4]:
# --- 3. CV folds: metrics across folds (compare models) ---
cv_folds = data["cv_folds"].copy()
if "fold" not in cv_folds.columns:
  cv_folds = cv_folds.reset_index()

fold_metrics = [c for c in ["accuracy", "f1", "roc_auc", "mae", "rmse", "r2"] if c in cv_folds.columns]
wilcoxon_cols = [c for c in cv_folds.columns if "wilcoxon" in c.lower()]
plot_metrics = [c for c in fold_metrics if cv_folds[c].notna().any()]

n_metrics = len(plot_metrics)
if n_metrics:
  fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 5), squeeze=False)
  for ax, metric in zip(axes.ravel(), plot_metrics):
    sns.lineplot(data=cv_folds, x="fold", y=metric, hue="model_label", marker="o", ax=ax)
    ax.set_title(f"CV {metric} by fold")
    ax.set_xlabel("Fold")
    ax.set_xticks(sorted(cv_folds["fold"].dropna().unique()))
    ax.legend(title="", fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
  fig.tight_layout()
  save_fig(fig, "03_cv_folds_lineplots.png")

# Heatmap per metric: model x fold
for metric in plot_metrics:
  pivot = cv_folds.pivot_table(index="model_short", columns="fold", values=metric, aggfunc="first")
  if pivot.empty:
    continue
  fig, ax = plt.subplots(figsize=(8, max(4, 0.5 * len(pivot))))
  sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax)
  ax.set_title(f"CV {metric} — model x fold")
  save_fig(fig, f"03_cv_folds_heatmap_{metric}.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_lineplots.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_heatmap_accuracy.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_heatmap_f1.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_heatmap_roc_auc.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_heatmap_mae.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_heatmap_rmse.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\03_cv_folds_heatmap_r2.png


In [5]:
# --- 4. CV balance across folds ---
bal = data["cv_balance"].copy()
if "fold" not in bal.columns and bal.index.name == "fold":
  bal = bal.reset_index()

balance_metrics = numeric_metric_cols(bal, exclude={"n_rows"})
if "n_rows" in bal.columns:
  balance_metrics = ["n_rows"] + [c for c in balance_metrics if c != "n_rows"]

for metric in balance_metrics[:4]:  # limit to main balance cols
  fig, ax = plt.subplots(figsize=(12, 5))
  sns.barplot(data=bal, x="fold", y=metric, hue="model_label", ax=ax)
  ax.set_title(f"CV fold balance: {metric}")
  ax.set_xlabel("Fold")
  ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
  save_fig(fig, f"04_cv_balance_{metric}.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\04_cv_balance_n_rows.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\04_cv_balance_mean_phq8.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\04_cv_balance_median_phq8.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\04_cv_balance_control.png


In [6]:
# --- 5. CV summary: compare models (mean ± std) ---
cv_sum = data["cv_summary"]

mean_cols = [c for c in cv_sum.columns if c.endswith("_mean") and pd.api.types.is_numeric_dtype(cv_sum[c])]
std_cols = {c.replace("_mean", "_std"): c for c in mean_cols}

for mean_col in mean_cols:
  std_col = mean_col.replace("_mean", "_std")
  plot_df = cv_sum[["model_label", "dataset", mean_col]].copy()
  plot_df["value"] = plot_df[mean_col]
  if std_col in cv_sum.columns:
    plot_df["err"] = cv_sum[std_col]
  else:
    plot_df["err"] = 0

  fig, ax = plt.subplots(figsize=(12, 5))
  x = np.arange(len(plot_df))
  colors = sns.color_palette("Set2", len(plot_df))
  ax.bar(x, plot_df["value"], yerr=plot_df["err"], capsize=4, color=colors)
  ax.set_xticks(x)
  ax.set_xticklabels(plot_df["model_label"], rotation=35, ha="right")
  ax.set_title(f"CV summary: {mean_col}")
  ax.set_ylabel(mean_col)
  save_fig(fig, f"05_cv_summary_{mean_col}.png")

# Facet by dataset
if mean_cols:
  long = cv_sum.melt(
    id_vars=["model_label", "dataset", "model_short"],
    value_vars=mean_cols,
    var_name="metric",
    value_name="value",
  )
  g = sns.catplot(
    data=long,
    kind="bar",
    x="model_short",
    y="value",
    hue="dataset",
    col="metric",
    col_wrap=3,
    height=4,
    aspect=1.1,
    sharey=False,
  )
  g.fig.suptitle("CV summary metrics by dataset and model", y=1.02)
  for ax in g.axes.ravel():
    ax.tick_params(axis="x", rotation=30)
  save_fig(g.fig, "05_cv_summary_all_metrics.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_accuracy_mean.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_f1_mean.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_roc_auc_mean.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_mae_mean.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_rmse_mean.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_r2_mean.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\05_cv_summary_all_metrics.png


In [7]:
def plot_subgroup_sheet(df, sheet_name, filename_prefix, normalize_gender=False):
  """Bar plots for gender / age / intersection subgroup metrics."""
  metrics = [c for c in ["accuracy", "f1", "roc_auc"] if c in df.columns]
  if not metrics:
    return

  group_col = "group" if "group" in df.columns else df.columns[2]
  if normalize_gender:
    df = combine_gender_subgroups(df, group_col)

  long = df.melt(
    id_vars=["model_label", "dataset", "model_short", group_col],
    value_vars=metrics,
    var_name="metric",
    value_name="score",
  )
  long = long.rename(columns={group_col: "subgroup"})

  subgroup_order = None
  if normalize_gender:
    present = df[group_col].dropna().unique().tolist()
    if all(" " not in str(g) for g in present):
      subgroup_order = [g for g in ["female", "male"] if g in present]
    else:
      age_order = ["young", "middle", "older"]
      gender_order = ["female", "male"]
      subgroup_order = [
        f"{age} {gender}"
        for age in age_order
        for gender in gender_order
        if f"{age} {gender}" in present
      ]

  for metric in metrics:
    sub = long[long["metric"] == metric]
    g = sns.catplot(
      data=sub,
      kind="bar",
      x="subgroup",
      y="score",
      hue="model_short",
      col="dataset",
      col_wrap=2,
      height=5,
      aspect=1.3,
      sharey=True,
      order=subgroup_order,
    )
    g.fig.suptitle(f"{sheet_name}: {metric} by subgroup", y=1.02)
    for ax in g.axes.ravel():
      ax.tick_params(axis="x", rotation=40)
    save_fig(g.fig, f"{filename_prefix}_{metric}.png")

  # Heatmap: model_label x subgroup for accuracy
  if "accuracy" in metrics:
    pivot = df.pivot_table(index="model_label", columns=group_col, values="accuracy", aggfunc="first")
    if normalize_gender and subgroup_order:
      cols = [c for c in subgroup_order if c in pivot.columns]
      pivot = pivot[cols]
    fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1] * 0.8), max(4, pivot.shape[0] * 0.45)))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"{sheet_name}: accuracy by model and subgroup")
    save_fig(fig, f"{filename_prefix}_accuracy_heatmap.png")


plot_subgroup_sheet(data["subgroup_gender"], "Gender subgroups", "06_subgroup_gender", normalize_gender=True)
plot_subgroup_sheet(data["subgroup_age"], "Age subgroups", "07_subgroup_age")
plot_subgroup_sheet(data["subgroup_gender_age"], "Gender x age subgroups", "08_subgroup_gender_age", normalize_gender=True)

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\06_subgroup_gender_accuracy.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\06_subgroup_gender_f1.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\06_subgroup_gender_roc_auc.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\06_subgroup_gender_accuracy_heatmap.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\07_subgroup_age_accuracy.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\07_subgroup_age_f1.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\07_subgroup_age_roc_auc.png
Saved: C:\Users\janku\Documents\KCL\Resear

In [8]:
# --- 6. Fairness metrics (AIF360) ---
fair = data["fairness"]
fair_metrics = [c for c in ["disparate_impact", "statistical_parity_difference"] if c in fair.columns]

melted = fair.melt(
  id_vars=["model_label", "dataset", "model_short"],
  value_vars=fair_metrics,
  var_name="fairness_metric",
  value_name="value",
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=melted, x="model_label", y="value", hue="fairness_metric", ax=ax)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="DI = 1 (parity)")
ax.axhline(0.0, color="black", linestyle=":", linewidth=1)
ax.set_title("Fairness metrics on held-out predictions")
ax.tick_params(axis="x", rotation=35)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_fig(fig, "09_fairness_combined.png")

g = sns.catplot(
  data=melted,
  kind="bar",
  x="model_short",
  y="value",
  hue="fairness_metric",
  col="dataset",
  height=4,
  aspect=1.2,
)
g.fig.suptitle("Fairness by dataset and model", y=1.02)
save_fig(g.fig, "09_fairness_by_dataset.png")

Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\09_fairness_combined.png
Saved: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel\09_fairness_by_dataset.png


WindowsPath('C:/Users/janku/Documents/KCL/Research Project/Research Project/results/figures/gradboost_from_excel/09_fairness_by_dataset.png')

In [9]:
# --- 7. Overview dashboard: CV test metrics + fairness (classifiers only) ---
test_sum = data.get("test_summary", pd.DataFrame())
if not test_sum.empty:
  cls_metrics = [c for c in ["accuracy", "f1", "roc_auc"] if c in test_sum.columns]
  reg_metrics = [c for c in ["mae", "rmse", "r2"] if c in test_sum.columns]
  use_metrics = [c for c in cls_metrics + reg_metrics if test_sum[c].notna().any()]

  long = test_sum.melt(
    id_vars=["model_label", "dataset", "model_short"],
    value_vars=use_metrics,
    var_name="metric",
    value_name="value",
  )
  g = sns.catplot(
    data=long,
    kind="bar",
    x="model_short",
    y="value",
    col="metric",
    hue="dataset",
    col_wrap=3,
    height=4,
    aspect=1.1,
    sharey=False,
  )
  g.fig.suptitle("Held-out test performance", y=1.02)
  save_fig(g.fig, "10_test_summary.png")

print(f"\nAll figures saved under: {FIG_DIR.resolve()}")


All figures saved under: C:\Users\janku\Documents\KCL\Research Project\Research Project\results\figures\gradboost_from_excel
